# 04 — numbering & germline (ANARCI 직접 실행)

> 본문: [`04_numbering.md`](04_numbering.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 9초** (실측)

**이 노트북은 도구를 직접 돌립니다.** ANARCI 를 **직접 돌려** IMGT·Chothia numbering CSV 를 `my_run/` 에 만들고, 커밋된 결과와 대조해요.
각 절은 **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서예요.
내가 만든 결과는 `my_run/` 에 쌓이고, 저장소에 커밋된 `data/` 는 **대조군(레퍼런스)** 으로만 씁니다.
어느 단계를 건너뛰거나 실패해도 `resolve()` 가 `data/` 로 폴백해서 뒤 절이 계속 돌아가요.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

**Colab**: 이 셀을 그대로 실행하면 클론 → 챕터 폴더 이동 → 필요한 도구 설치까지 한 번에 끝나요.
**로컬**: 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "04_numbering"
PIP_PKGS = "pandas matplotlib anarci abnumber"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = True        # ANARCI 계열은 HMMER 의 hmmscan 실행파일이 필요해요 (pip 로는 안 깔려요)
PIN_TRANSFORMERS = None    # IgFold 체크포인트가 요구하는 transformers 버전(없으면 None)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

if NEED_HMMER and shutil.which("hmmscan") is None:
    # ANARCI 는 내부적으로 hmmscan 을 호출해요. pip install anarci 만으로는 안 깔려요.
    if IN_COLAB:
        _run("apt-get -qq update")                       # 인덱스가 낡으면 install 이 404 로 죽는다
        _run("apt-get -qq install -y hmmer")             # ← ANARCI 가 부르는 hmmscan
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if "igfold" in PIP_PKGS and importlib.util.find_spec("pkg_resources") is None:
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 해요.
    _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
    importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


if PIN_TRANSFORMERS:
    # IgFold 체크포인트에는 옛 transformers 의 토크나이저 객체가 pickle 돼 있어요.
    # 최신 transformers(5.x) 로는 unpickle 이 실패해서(Trie/BasicTokenizer 없음) 버전을 맞춥니다.
    _ver = subprocess.run([sys.executable, "-c",
                           "import transformers;print(transformers.__version__)"],
                          capture_output=True, text=True).stdout.strip()
    if not _ver.startswith("4."):
        print(f"[transformers {_ver or 'none'} → {PIN_TRANSFORMERS}] IgFold 체크포인트 호환 버전으로 맞춥니다")
        _run(f'"{sys.executable}" -m pip -q install "transformers=={PIN_TRANSFORMERS}"')

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — ANARCI numbering (본문 4.1)

```bash
ANARCI -i data/demo_mab.fa -s imgt --assign_germline --csv --outfile my_run/demo_imgt
ANARCI -i data/demo_mab.fa -s chothia --csv --outfile my_run/demo_chothia
```
사슬별 CSV(`..._H.csv`, `..._KL.csv`)가 생깁니다. 각 numbering 위치가 **컬럼 하나**가 돼요.

In [ ]:
ok_imgt = run_tool(["ANARCI", "-i", "data/demo_mab.fa", "-s", "imgt",
                    "--assign_germline", "--csv", "--outfile", "my_run/demo_imgt"])
ok_chot = run_tool(["ANARCI", "-i", "data/demo_mab.fa", "-s", "chothia",
                    "--csv", "--outfile", "my_run/demo_chothia"])
import pathlib
print("\n생성된 파일:", sorted(p.name for p in pathlib.Path("my_run").glob("*.csv")))

## 2) 내 결과 확인 — germline 할당 (본문 4.2)

In [ ]:
import pandas as pd
for label, f in [("Heavy", "demo_imgt_H.csv"), ("Light", "demo_imgt_KL.csv")]:
    r = pd.read_csv(resolve(f)).iloc[0]
    print(f"{label:6s}: chain_type={r['chain_type']}  V={r['v_gene']} ({r['v_identity']*100:.0f}%)  "
          f"J={r['j_gene']} ({r['j_identity']*100:.0f}%)  score={r['score']}")
print("\n[심화] Heavy V=IGHV14-4 → 마우스 germline! → Ch.05 humanization 대상")

## 3) 내 결과 확인 — IMGT vs Chothia CDR-H1 경계 (본문 4.3)

In [ ]:
import pandas as pd
def occupied(f, lo, hi):
    r = pd.read_csv(f).iloc[0]
    cols = [c for c in r.index if str(c).strip().isdigit() and lo <= int(c) <= hi]
    return [c for c in cols if str(r[c]) not in ("nan", "-", "")]

imgt_h  = resolve("demo_imgt_H.csv")
chot_h  = resolve("demo_chothia_H.csv")
print("IMGT    CDR-H1 (27-38):", len(occupied(imgt_h, 27, 38)), "잔기")
print("Chothia CDR-H1 (26-32):", len(occupied(chot_h, 26, 32)), "잔기")
print("\n[주의] scheme 마다 경계가 달라요 → 보고서엔 항상 scheme 명시!")

# CDR 서열도 뽑아 두면 뒤 챕터(08·09)에서 그대로 씁니다.
from abnumber import Chain
seqs, name = {}, None
for line in open("data/demo_mab.fa"):
    line = line.strip()
    if line.startswith(">"): name = line[1:].split()[0]; seqs[name] = ""
    elif name: seqs[name] += line
rows = []
for sid, seq in seqs.items():
    ch = Chain(seq, scheme="imgt")
    rows.append({"id": sid, "chain_type": ch.chain_type, "cdr1": ch.cdr1_seq,
                 "cdr2": ch.cdr2_seq, "cdr3": ch.cdr3_seq, "cdr3_len": len(ch.cdr3_seq)})
cdrs = pd.DataFrame(rows)
cdrs.to_csv("my_run/demo_cdrs.csv", index=False)
display(cdrs)

## 4) 레퍼런스 대조 — 커밋된 ANARCI 결과와 같은가

`data/demo_imgt_H.csv` 는 이 저장소를 만들 때 ANARCI 로 돌려 커밋해 둔 대조군이에요.
내가 방금 만든 CSV와 **셀 단위로** 비교합니다.

In [ ]:
import pandas as pd, pathlib
def compare(fname):
    mine_p, ref_p = pathlib.Path("my_run")/fname, pathlib.Path("data")/fname
    if not mine_p.exists():
        print(f"{fname}: my_run 산출물 없음 → 대조 건너뜀"); return
    mine, ref = pd.read_csv(mine_p), pd.read_csv(ref_p)
    common = [c for c in ref.columns if c in mine.columns]
    same = mine[common].astype(str).equals(ref[common].astype(str))
    print(f"{fname}: 공통 컬럼 {len(common)}개 / 값 완전 일치 = {same}")
    if not same:
        for c in common:
            if not mine[c].astype(str).equals(ref[c].astype(str)):
                print("   차이 컬럼:", c, "| 내 결과:", mine[c].tolist(), "| 레퍼런스:", ref[c].tolist())
for f in ["demo_imgt_H.csv", "demo_imgt_KL.csv", "demo_chothia_H.csv", "demo_chothia_KL.csv"]:
    compare(f)

> 다음 → 본문 [05. humanness & humanization](../05_humanness/05_humanness.md)